# 01 — Extraire les stations Citiz (GBFS)

**Rôle : pipeline, étape 1** (puis `citiz` 03 et `lakehouse` 01). Produit les CSV bruts lus par les notebooks 02 et 03 et par le Lakehouse.

**Objectif** : récupérer toutes les stations Citiz publiées sur le Point d'accès national (transport.data.gouv.fr) et les enregistrer en CSV bruts.

**GBFS** (*General Bikeshare Feed Specification*) est le format standard des services partagés (vélos, trottinettes, autopartage). Un flux GBFS est un fichier `gbfs.json` qui liste des sous-flux :
- `station_information` : où sont les stations (position, adresse, capacité) ;
- `station_status` : ce qui est disponible **maintenant** (nombre de véhicules) ;
- `vehicle_types` : les modèles de véhicules (marque, motorisation).

Citiz publie **un flux par réseau régional** (15 au total).

**Étapes**
1. Interroger l'API du Point d'accès national et garder les jeux de données Citiz.
2. Lire le `gbfs.json` de chaque flux pour trouver ses sous-flux.
3. Lire stations, disponibilités et types de véhicules.
4. Nettoyer et dédoublonner.
5. Écrire 4 CSV dans `citiz/data/raw/gbfs/`.

**Historique** : GBFS ne donne que l'état instantané. Chaque exécution **ajoute** un instantané à `citiz_station_status.csv` ; relancer régulièrement ce notebook est le seul moyen d'observer l'usage des stations.

In [1]:
from pathlib import Path

try:
    spark  # noqa: F821  (defined automatically in Fabric)
    IN_FABRIC = True
except NameError:
    IN_FABRIC = False

if IN_FABRIC:
    output_directory = Path("/lakehouse/default/Files/raw/gbfs")
else:
    from pathlib import Path

    current_directory = Path.cwd().resolve()
    project_directory = None
    for directory in [current_directory, *current_directory.parents]:
        if (directory / "citiz" / "AGENTS.md").exists():
            project_directory = directory
            break
    if project_directory is None:
        raise FileNotFoundError("Open this notebook from the repository directory.")

    module_directory = project_directory / "citiz"
    gbfs_directory = module_directory / "data" / "raw" / "gbfs"
    boundaries_directory = module_directory / "data" / "raw" / "boundaries"
    processed_directory = module_directory / "data" / "processed"
    for directory in [gbfs_directory, boundaries_directory, processed_directory]:
        directory.mkdir(parents=True, exist_ok=True)
    output_directory = gbfs_directory
output_directory.mkdir(parents=True, exist_ok=True)

import re
import json
import time
from datetime import datetime, timezone
import requests
import pandas as pd

PAN_DATASETS_URL = "https://transport.data.gouv.fr/api/datasets"
SNAPSHOT_TS = datetime.now(timezone.utc)

session = requests.Session()
session.headers.update({"User-Agent": "hackathon-citiz/1.0"})


def get_json(url, retries=3, timeout=30):
    """GET with a few retries; return None if the resource stays unreachable."""
    for attempt in range(retries):
        try:
            response = session.get(url, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except Exception as error:
            if attempt == retries - 1:
                print(f"  failed {url}: {error}")
                return None
            time.sleep(2 ** attempt)

## 1. Trouver les jeux Citiz dans l'API du Point d'accès national

L'API renvoie tous les jeux de données en une seule liste. On garde ceux dont le titre, l'identifiant ou le producteur contient « citiz », puis leurs ressources au format `gbfs`.
`original_url` (URL du producteur) est préférée à `url` (redirection data.gouv.fr).

La colonne `source` donne un identifiant court au flux (`provider_12`, sinon l'identifiant du jeu). Elle sert ensuite à construire `station_uid`.

In [2]:
datasets = get_json(PAN_DATASETS_URL, timeout=120)
print(f"{len(datasets)} datasets in the national access point")


def is_citiz(dataset):
    publisher = dataset.get("publisher") or {}
    text = " ".join(str(value) for value in [dataset.get("title"), dataset.get("slug"), publisher.get("name")] if value)
    return "citiz" in text.lower()


def source_id(row):
    match = re.search(r"provider/(\d+)", row.gbfs_url or "")
    if match:
        return f"provider_{match.group(1)}"
    return row.dataset_slug


rows = []
for dataset in filter(is_citiz, datasets):
    for resource in dataset.get("resources", []):
        if (resource.get("format") or "").lower() != "gbfs":
            continue
        rows.append({
            "dataset_id": dataset.get("id"),
            "dataset_slug": dataset.get("slug"),
            "dataset_title": dataset.get("title"),
            "publisher": (dataset.get("publisher") or {}).get("name"),
            "page_url": dataset.get("page_url"),
            "licence": dataset.get("licence"),
            "resource_title": resource.get("title"),
            "gbfs_url": resource.get("original_url") or resource.get("url"),
            "is_available": resource.get("is_available"),
        })

feeds = pd.DataFrame(rows).drop_duplicates("gbfs_url").reset_index(drop=True)
feeds["source"] = feeds.apply(source_id, axis=1)
print(f"{feeds.dataset_slug.nunique()} Citiz datasets, {len(feeds)} GBFS feeds")
feeds[["source", "dataset_title", "gbfs_url", "is_available"]]

801 datasets in the national access point
15 Citiz datasets, 15 GBFS feeds


,source,dataset_title,gbfs_url,is_available
0,provider_12,Autopartage Citiz Développement,https://backend.citiz.fr/public/provider/12/gb...,True
1,provider_5,Autopartage Citiz Auvergne Rhône Alpes (AURA),https://backend.citiz.fr/public/provider/5/gbf...,True
2,provider_1,Autopartage Citiz Grand Est,https://backend.citiz.fr/public/provider/1/gbf...,True
3,offre-et-disponibilite-temps-reel-du-service-c...,Autopartage Citiz Métropole de Lyon,https://data.grandlyon.com/files/rdata/lpa_mob...,True
4,provider_6,Autopartage Citiz Provence,https://backend.citiz.fr/public/provider/6/gbf...,True
5,provider_8,Autopartage Citiz Occitanie,https://backend.citiz.fr/public/provider/8/gbf...,True
6,provider_16,Autopartage Citiz Haut de France,https://backend.citiz.fr/public/provider/16/gb...,True
7,provider_15,Autopartage Citiz Nantes Métropole,https://backend.citiz.fr/public/provider/15/gb...,True
8,provider_4,Autopartage Citiz Gironde,https://backend.citiz.fr/public/provider/4/gbf...,True
9,provider_19,Autopartage Citiz Rennes Métropole,https://backend.citiz.fr/public/provider/19/gb...,True


## 2. Lire le `gbfs.json` de chaque flux

La structure diffère selon la version : GBFS 3.0 range les sous-flux dans `data.feeds`, GBFS 2.x dans `data.<langue>.feeds`. On gère les deux.

In [3]:
def list_feeds(gbfs):
    data = (gbfs or {}).get("data", {})
    if "feeds" in data:
        sub_feeds = data["feeds"]
    else:
        language = "fr" if "fr" in data else next(iter(data), None)
        sub_feeds = data.get(language, {}).get("feeds", []) if language else []
    return {feed["name"]: feed["url"] for feed in sub_feeds}


feed_urls = {}
for row in feeds.itertuples():
    gbfs = get_json(row.gbfs_url)
    sub_feeds = list_feeds(gbfs)
    feed_urls[row.source] = sub_feeds
    feeds.loc[row.Index, "gbfs_version"] = (gbfs or {}).get("version")
    feeds.loc[row.Index, "feeds"] = ",".join(sorted(sub_feeds))
    print(f"{row.source:<20} v{(gbfs or {}).get('version')}  {sorted(sub_feeds)}")

provider_12          v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_5           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_1           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']


offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_6           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_8           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_16          v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_15          v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans

provider_2           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_7           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_9           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_18          v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']


## 3. Lire stations, disponibilités et types de véhicules

Deux particularités de GBFS 3.0 :
- les textes (`name`, `city`…) sont des listes `[{text, language}]` ; on garde le français ;
- `last_reported` est une date texte (RFC 3339), alors que GBFS 2.x donne un nombre de secondes.

In [4]:
def text_of(value):
    """Localised GBFS 3.0 field -> plain text (French first)."""
    if isinstance(value, list):
        french = [item.get("text") for item in value if item.get("language", "").startswith("fr")]
        if french:
            return french[0]
        return value[0].get("text") if value else None
    return value


def to_timestamp(value):
    if value is None:
        return pd.NaT
    if isinstance(value, (int, float)):
        return pd.to_datetime(value, unit="s", utc=True)
    return pd.to_datetime(value, utc=True, errors="coerce")


def read_feed(source, name, key):
    url = feed_urls.get(source, {}).get(name)
    if not url:
        return []
    payload = get_json(url)
    return ((payload or {}).get("data") or {}).get(key, [])


def json_or_none(value):
    return json.dumps(value) if value else None


station_rows, status_rows, vehicle_type_rows = [], [], []
for source in feeds.source:
    for station in read_feed(source, "station_information", "stations"):
        station_rows.append({
            "source": source,
            "station_id": str(station.get("station_id")),
            "name": text_of(station.get("name")),
            "short_name": text_of(station.get("short_name")),
            "lat": station.get("lat"),
            "lon": station.get("lon"),
            "address": station.get("address"),
            "post_code": station.get("post_code"),
            "city": text_of(station.get("city")),
            "capacity": station.get("capacity"),
            "is_virtual_station": station.get("is_virtual_station"),
            "vehicle_types_capacity": json_or_none(station.get("vehicle_types_capacity")),
        })
    for status in read_feed(source, "station_status", "stations"):
        status_rows.append({
            "source": source,
            "station_id": str(status.get("station_id")),
            "num_vehicles_available": status.get("num_vehicles_available", status.get("num_bikes_available")),
            "num_vehicles_disabled": status.get("num_vehicles_disabled"),
            "num_docks_available": status.get("num_docks_available"),
            "is_installed": status.get("is_installed"),
            "is_renting": status.get("is_renting"),
            "is_returning": status.get("is_returning"),
            "last_reported": to_timestamp(status.get("last_reported")),
            "vehicle_types_available": json_or_none(status.get("vehicle_types_available")),
            "snapshot_ts": SNAPSHOT_TS,
        })
    for vehicle_type in read_feed(source, "vehicle_types", "vehicle_types"):
        vehicle_type_rows.append({
            "source": source,
            "vehicle_type_id": str(vehicle_type.get("vehicle_type_id")),
            "name": text_of(vehicle_type.get("name")),
            "make": text_of(vehicle_type.get("make")),
            "model": text_of(vehicle_type.get("model")),
            "form_factor": vehicle_type.get("form_factor"),
            "propulsion_type": vehicle_type.get("propulsion_type"),
            "rider_capacity": vehicle_type.get("rider_capacity"),
            "max_range_meters": vehicle_type.get("max_range_meters"),
        })

STATUS_COLUMNS = ["source", "station_id", "num_vehicles_available", "num_vehicles_disabled", "num_docks_available",
                  "is_installed", "is_renting", "is_returning", "last_reported", "vehicle_types_available", "snapshot_ts"]
stations = pd.DataFrame(station_rows)
status = pd.DataFrame(status_rows, columns=STATUS_COLUMNS)
vehicle_types = pd.DataFrame(vehicle_type_rows)
print(f"stations: {len(stations)} | statuses: {len(status)} | vehicle types: {len(vehicle_types)}")
stations.head()

stations: 1615 | statuses: 1615 | vehicle types: 337


,source,station_id,name,short_name,lat,lon,address,post_code,city,capacity,is_virtual_station,vehicle_types_capacity
0,provider_12,43,Halles,None,47.392776,0.679685,34 Rue de la Grosse Tour,None,None,3,False,None
1,provider_12,44,Constantine-Commerce,None,47.395157,0.684126,NaN,None,None,2,False,None
2,provider_12,45,Cathédrale-Sicard,None,47.394527,0.693761,17 place François Sicard,None,None,2,False,None
3,provider_12,46,Gare,None,47.390404,0.694493,NaN,None,None,2,False,None
4,provider_12,47,Jean Jaurès (16 bd Beranger),None,47.389912,0.687188,NaN,None,None,2,False,None


## 4. Nettoyer et dédoublonner

- On retire les stations sans coordonnées ou hors de la France métropolitaine (Corse incluse) et des DROM.
- Certains flux peuvent se recouper (Auvergne-Rhône-Alpes et Métropole de Lyon, « Citiz Développement »). Une station est considérée en double si elle a le même nom et la même position arrondie à environ 10 m. On garde une ligne et la liste de ses flux dans `sources`.
- `station_uid` = `source:station_id` est la **clé de jointure** utilisée par toute l'équipe : un même `station_id` existe dans plusieurs réseaux.
- La dernière disponibilité connue est jointe à la station, pour les cartes.

In [5]:
stations["lat"] = pd.to_numeric(stations["lat"], errors="coerce")
stations["lon"] = pd.to_numeric(stations["lon"], errors="coerce")
initial_count = len(stations)

in_mainland = stations.lat.between(41, 51.5) & stations.lon.between(-5.5, 10)
OVERSEAS_BOXES = [
    (15.8, 16.6, -61.9, -60.9),   # Guadeloupe
    (14.3, 14.9, -61.3, -60.8),   # Martinique
    (2.0, 6.0, -54.7, -51.5),     # French Guiana
    (-21.5, -20.8, 55.1, 55.9),   # Reunion
    (-13.1, -12.6, 44.9, 45.4),   # Mayotte
]
in_overseas = pd.Series(False, index=stations.index)
for lat_min, lat_max, lon_min, lon_max in OVERSEAS_BOXES:
    in_overseas |= stations.lat.between(lat_min, lat_max) & stations.lon.between(lon_min, lon_max)
stations = stations[stations.lat.notna() & (in_mainland | in_overseas)].copy()
print(f"{initial_count - len(stations)} stations removed (missing or out-of-range coordinates)")

stations["dedup_key"] = (stations.lat.round(4).astype(str) + "_" + stations.lon.round(4).astype(str)
                         + "_" + stations.name.fillna("").str.lower().str.strip())
sources_by_station = stations.groupby("dedup_key")["source"].agg(lambda values: ",".join(sorted(set(values))))
stations = stations.drop_duplicates("dedup_key").copy()
stations["sources"] = stations.dedup_key.map(sources_by_station)
stations["station_uid"] = stations.source + ":" + stations.station_id
stations["extracted_at"] = SNAPSHOT_TS

status["station_uid"] = status.source + ":" + status.station_id
stations = stations.merge(status[["station_uid", "num_vehicles_available", "is_renting"]], on="station_uid", how="left")
stations["geometry_wkt"] = "POINT(" + stations.lon.astype(str) + " " + stations.lat.astype(str) + ")"

print(f"{len(stations)} unique Citiz stations")
stations.groupby("source").size().sort_values(ascending=False)

0 stations removed (missing or out-of-range coordinates)
1615 unique Citiz stations


source
provider_5                                                                             412
provider_1                                                                             290
offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon    200
provider_8                                                                             142
provider_4                                                                             100
provider_6                                                                              97
provider_16                                                                             80
provider_19                                                                             66
provider_12                                                                             64
provider_2                                                                              60
provider_7                                                                         

## 5. Écrire les CSV bruts

| Fichier | Contenu | Écriture |
|---|---|---|
| `citiz_flux.csv` | un flux GBFS par ligne | remplacé |
| `citiz_stations.csv` | une station par ligne | remplacé |
| `citiz_vehicle_types.csv` | modèles de véhicules par réseau | remplacé |
| `citiz_station_status.csv` | disponibilités à l'instant de l'extraction | **ajouté** à l'historique |

Dans Microsoft Fabric (variable `spark` définie), les fichiers vont dans `Files/raw/gbfs/` du Lakehouse attaché.

In [6]:
def write_raw(df, name, mode):
    """Write <name>.csv; mode "append" adds rows without repeating the header."""
    if df.empty:
        print(f"{name}: nothing to write")
        return
    path = output_directory / f"{name}.csv"
    append = mode == "append" and path.exists()
    df.to_csv(path, index=False, mode="a" if append else "w", header=not append)
    print(f"{name} -> {path.name} ({len(df)} rows, {mode})")


write_raw(feeds, "citiz_flux", "overwrite")
write_raw(stations.drop(columns=["dedup_key"]), "citiz_stations", "overwrite")
write_raw(vehicle_types, "citiz_vehicle_types", "overwrite")
write_raw(status, "citiz_station_status", "append")

citiz_flux -> citiz_flux.csv (15 rows, overwrite)
citiz_stations -> citiz_stations.csv (1615 rows, overwrite)
citiz_vehicle_types -> citiz_vehicle_types.csv (337 rows, overwrite)
citiz_station_status -> citiz_station_status.csv (1615 rows, append)


## 6. Vérification

Relire les fichiers écrits : clé unique, nombre d'instantanés accumulés.

In [7]:
saved_stations = pd.read_csv(output_directory / "citiz_stations.csv", dtype={"station_id": "string"})
assert saved_stations["station_uid"].is_unique
history = pd.read_csv(output_directory / "citiz_station_status.csv", usecols=["snapshot_ts"])
print(f"{len(saved_stations)} stations saved")
print(f"{history.snapshot_ts.nunique()} snapshot(s), from {history.snapshot_ts.min()} to {history.snapshot_ts.max()}")

1615 stations saved
3 snapshot(s), from 2026-10-19 14:43:54.580579+00:00 to 2026-10-20 08:43:42.155384+00:00
